# SAE Operational Fingerprint — Smoky Experiment

**Goal:** Minimum viable test of the constant-trace hypothesis.

We collect SAE feature activations from GPT-2 Small on arithmetic prompts,
then check:
1. **Operand-range shift** — does the trace change when operands go from [0,20] → [100,200]?
2. **Format shift** — does the trace change across surface formats (`3+5`, `three plus five`, `sum of 3 and 5`)?

A true operational fingerprint should survive both shifts.
A surface-token fingerprint won't.

We also compute **within-operation trace variance** as a function of operand range
and correlate it with per-range accuracy — the core of the constant-trace robustness metric.

SAE: Joseph Bloom's GPT-2 Small SAE via `sae_lens` (layer 8, residual stream).

In [ ]:
# ── 0. Install (Colab only) ──────────────────────────────────────────────────
!pip install -q sae_lens transformer_lens

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import random
import itertools
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr
from transformer_lens import HookedTransformer
from sae_lens import SAE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ── 2. Load GPT-2 Small + SAE ─────────────────────────────────────────────────
# SAE: Bloom et al. GPT-2 Small, layer 8, residual stream post.
# Release id comes from sae_lens pretrained SAE registry.
model = HookedTransformer.from_pretrained("gpt2", center_writing_weights=False)
model.eval()
model.to(DEVICE)

sae, cfg_dict, _ = SAE.from_pretrained(
    release="gpt2-small-res-jb",   # Joseph Bloom GPT-2 Small residual stream SAEs
    sae_id="blocks.8.hook_resid_pre",
    device=DEVICE,
)
sae.eval()
print(f"SAE dict size: {sae.cfg.d_sae}")

In [ ]:
# ── 3. Data generation ────────────────────────────────────────────────────────
RANGES = {
    "tiny":   (0,   10),   # memorizable
    "small":  (0,   20),
    "medium": (20, 100),
    "large":  (100, 200),  # OOD for GPT-2
}

FORMATS = {
    "symbolic":  lambda a, b: f"{a}+{b}=",
    "spaced":    lambda a, b: f"{a} + {b} =",
    "worded":    lambda a, b: f"What is {a} plus {b}? Answer:",
}

def make_addition_prompts(lo, hi, n=80, fmt_key="symbolic", seed=42):
    rng = random.Random(seed)
    fmt = FORMATS[fmt_key]
    pairs = [(rng.randint(lo, hi), rng.randint(lo, hi)) for _ in range(n)]
    return [(a, b, a + b, fmt(a, b)) for a, b in pairs]

# Sanity check
sample = make_addition_prompts(0, 20, n=3)
for a, b, c, prompt in sample:
    print(f"  {prompt!r}  → {c}")

In [ ]:
# ── 4. Feature extraction ─────────────────────────────────────────────────────
# We extract SAE features at the *last token* position of each prompt,
# as that's where the model "decides" the answer.

@torch.no_grad()
def get_sae_features(prompts: list[str], batch_size: int = 16) -> np.ndarray:
    """Returns (N, d_sae) float32 feature activation matrix."""
    all_feats = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]
        tokens = model.to_tokens(batch, prepend_bos=True)  # (B, T)
        _, cache = model.run_with_cache(tokens, names_filter="blocks.8.hook_resid_pre")
        resid = cache["blocks.8.hook_resid_pre"]  # (B, T, d_model)
        last = resid[:, -1, :]                    # (B, d_model) — last token
        feats = sae.encode(last)                  # (B, d_sae)
        all_feats.append(feats.float().cpu().numpy())
    return np.concatenate(all_feats, axis=0)

# Collect data for all range × format combinations
records = []
feature_matrix = {}   # key → (N, d_sae)

for range_name, (lo, hi) in RANGES.items():
    for fmt_name in FORMATS:
        data = make_addition_prompts(lo, hi, n=80, fmt_key=fmt_name)
        prompts = [d[3] for d in data]
        feats = get_sae_features(prompts)
        key = (range_name, fmt_name)
        feature_matrix[key] = feats
        for (a, b, c, _), f in zip(data, feats):
            records.append({
                "range": range_name, "fmt": fmt_name,
                "a": a, "b": b, "expected": c,
                "lo": lo, "hi": hi,
            })
        print(f"  {range_name:8s} × {fmt_name:10s}  feats shape: {feats.shape}")

df = pd.DataFrame(records)

In [ ]:
# ── 5. Accuracy measurement ───────────────────────────────────────────────────
# GPT-2 is weak at arithmetic, but we need to know *where* it fails.

@torch.no_grad()
def check_accuracy(data: list, fmt_key: str) -> list[bool]:
    """
    Greedy decode one token after each prompt,
    check if it matches str(expected).
    Only works for single-token answers (sums < ~100).
    """
    results = []
    for a, b, c, prompt in data:
        tokens = model.to_tokens(prompt, prepend_bos=True)
        logits = model(tokens)[0, -1, :]         # (d_vocab,)
        pred_token = logits.argmax().item()
        pred_str = model.tokenizer.decode([pred_token]).strip()
        results.append(pred_str == str(c))
    return results

acc_rows = []
for range_name, (lo, hi) in RANGES.items():
    for fmt_name in FORMATS:
        data = make_addition_prompts(lo, hi, n=80, fmt_key=fmt_name)
        correct = check_accuracy(data, fmt_name)
        acc = np.mean(correct)
        acc_rows.append({"range": range_name, "fmt": fmt_name, "accuracy": acc, "lo": lo, "hi": hi})
        print(f"  {range_name:8s} × {fmt_name:10s}  accuracy: {acc:.2%}")

df_acc = pd.DataFrame(acc_rows)

In [ ]:
# ── 6. Constant-trace metric: within-group variance ───────────────────────────
# For each (range, fmt) group, compute mean feature activation variance
# across instances — this is the "trace variance" (low = constant trace).

variance_rows = []
for (range_name, fmt_name), feats in feature_matrix.items():
    # Active features only (sparsity: ignore zeros)
    active_mask = feats.max(axis=0) > 0    # features that fire at all
    v = feats[:, active_mask].var(axis=0).mean()   # mean variance over active features
    v_all = feats.var(axis=0).mean()               # mean variance over all features
    lo, hi = RANGES[range_name]
    variance_rows.append({
        "range": range_name, "fmt": fmt_name,
        "variance_active": float(v),
        "variance_all": float(v_all),
        "lo": lo, "hi": hi,
    })

df_var = pd.DataFrame(variance_rows)
df_merged = df_acc.merge(df_var, on=["range", "fmt", "lo", "hi"])
print(df_merged[["range", "fmt", "accuracy", "variance_active"]].to_string(index=False))

In [ ]:
# ── 7. Plot: variance vs accuracy across operand ranges ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: variance × accuracy scatter (one point per range × fmt)
colors = {"symbolic": "steelblue", "spaced": "darkorange", "worded": "green"}
ax = axes[0]
for fmt_name, grp in df_merged.groupby("fmt"):
    ax.scatter(grp["variance_active"], grp["accuracy"],
               label=fmt_name, color=colors[fmt_name], s=80)
    for _, row in grp.iterrows():
        ax.annotate(row["range"], (row["variance_active"], row["accuracy"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=8)

ax.set_xlabel("Mean feature variance (active features)")
ax.set_ylabel("Accuracy")
ax.set_title("Trace variance vs. accuracy\n(constant-trace hypothesis)")
ax.legend()

# Right: accuracy by range for each format
ax2 = axes[1]
range_order = ["tiny", "small", "medium", "large"]
for fmt_name, grp in df_acc.groupby("fmt"):
    grp_sorted = grp.set_index("range").reindex(range_order)
    ax2.plot(range_order, grp_sorted["accuracy"], marker="o", label=fmt_name, color=colors[fmt_name])

ax2.set_xlabel("Operand range")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy by operand range × format")
ax2.legend()

plt.tight_layout()
plt.savefig("smoky_results.png", dpi=150)
plt.show()

# Spearman correlation: does variance predict accuracy?
rho, p = spearmanr(df_merged["variance_active"], df_merged["accuracy"])
print(f"\nSpearman ρ(variance, accuracy) = {rho:.3f}  p={p:.4f}")
print("Hypothesis: ρ < 0 (higher variance → lower accuracy)")

In [ ]:
# ── 8. Format invariance test ─────────────────────────────────────────────────
# For the same operand range, how similar are traces across formats?
# We compare mean feature vectors (centroids) between format pairs.
# A format-invariant fingerprint → high cosine similarity between centroids.

from numpy.linalg import norm

def cosine_sim(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-8))

fmt_pairs = list(itertools.combinations(FORMATS.keys(), 2))
sim_rows = []
for range_name in RANGES:
    centroids = {f: feature_matrix[(range_name, f)].mean(axis=0) for f in FORMATS}
    for fa, fb in fmt_pairs:
        sim = cosine_sim(centroids[fa], centroids[fb])
        sim_rows.append({"range": range_name, "fmt_a": fa, "fmt_b": fb, "cosine_sim": sim})

df_sim = pd.DataFrame(sim_rows)
print("Format invariance (cosine similarity of centroids, same operand range):")
print(df_sim.pivot_table(index="range", columns=["fmt_a", "fmt_b"], values="cosine_sim").to_string())

In [ ]:
# ── 9. PCA visualization of all traces ───────────────────────────────────────
# Stack all feature matrices, run PCA, colour by (range, fmt).
# If traces cluster by range more than by format → range-sensitive (bad for fingerprint).
# If traces cluster by format more than by range → format-sensitive (also bad).
# Ideal: tight cluster per operation regardless of range/format.

all_feats_list, labels_range, labels_fmt = [], [], []
for (range_name, fmt_name), feats in feature_matrix.items():
    all_feats_list.append(feats)
    labels_range.extend([range_name] * len(feats))
    labels_fmt.extend([fmt_name] * len(feats))

X = np.concatenate(all_feats_list, axis=0)
X_scaled = StandardScaler().fit_transform(X)
pcs = PCA(n_components=2).fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

range_colors = {"tiny": "navy", "small": "royalblue", "medium": "orange", "large": "red"}
fmt_markers = {"symbolic": "o", "spaced": "s", "worded": "^"}

for ax, labels, cmap_dict, title in [
    (axes[0], labels_range, range_colors, "PCA coloured by operand range"),
    (axes[1], labels_fmt,  colors,       "PCA coloured by format"),
]:
    unique = sorted(set(labels))
    for lbl in unique:
        mask = np.array(labels) == lbl
        ax.scatter(pcs[mask, 0], pcs[mask, 1], c=cmap_dict[lbl],
                   alpha=0.3, s=20, label=lbl)
    ax.set_title(title)
    ax.legend(markerscale=2, fontsize=8)

plt.tight_layout()
plt.savefig("smoky_pca.png", dpi=150)
plt.show()

In [ ]:
# ── 10. Top discriminating features: range shift ─────────────────────────────
# Which SAE features change most between tiny and large operand ranges?
# These are the candidates for being range-encoding (surface) vs operation-encoding.

TOPK = 20
range_order = ["tiny", "small", "medium", "large"]

# Use symbolic format as baseline
centroids_by_range = {
    r: feature_matrix[(r, "symbolic")].mean(axis=0) for r in range_order
}

delta = centroids_by_range["large"] - centroids_by_range["tiny"]
top_idx = np.argsort(np.abs(delta))[::-1][:TOPK]

print(f"Top {TOPK} features with largest shift (tiny → large operands):")
print(f"{'Feature':>8}  {'tiny':>8}  {'large':>8}  {'delta':>8}")
for idx in top_idx:
    print(f"{idx:>8}  {centroids_by_range['tiny'][idx]:>8.4f}  "
          f"{centroids_by_range['large'][idx]:>8.4f}  {delta[idx]:>8.4f}")

In [ ]:
# ── 11. Summary ───────────────────────────────────────────────────────────────
print("=" * 60)
print("SMOKY EXPERIMENT SUMMARY")
print("=" * 60)

print("\n[1] Accuracy by operand range (symbolic format):")
for r in ["tiny", "small", "medium", "large"]:
    row = df_acc[(df_acc["range"] == r) & (df_acc["fmt"] == "symbolic")].iloc[0]
    print(f"    {r:8s}  {row['accuracy']:.2%}")

print("\n[2] Constant-trace hypothesis:")
print(f"    Spearman ρ(variance, accuracy) = {rho:.3f}  p={p:.4f}")
conclusion = "SUPPORTED" if (rho < -0.3 and p < 0.1) else "NOT SUPPORTED"
print(f"    Verdict: {conclusion}")

print("\n[3] Format invariance (mean cosine sim of centroids, same range):")
mean_sim = df_sim.groupby(["fmt_a", "fmt_b"])["cosine_sim"].mean()
print(mean_sim.to_string())

print("\n[4] PCA cluster structure → see smoky_pca.png")
print("=" * 60)
print("Next step: if variance does NOT predict accuracy → surface fingerprint, pivot.")
print("If variance DOES predict accuracy → proceed to multiplication + Idea C (RSA).")